# Lesson 6: Coffea (workbook)

Open this notebook next to the slides and run what the instructor runs.
The empty cells are yours: change the cuts, break things, see what happens.

In [ ]:
import numpy as np
import awkward as ak
import hist
import matplotlib.pyplot as plt

from coffea import processor
from coffea.nanoevents import NanoEventsFactory, BaseSchema, NanoAODSchema
from coffea.analysis_tools import PackedSelection, Weights

In [ ]:
class SlimNanoAODSchema(NanoAODSchema):
    warn_missing_crossrefs = False


access_log = []
events = NanoEventsFactory.from_root(
    {"../data/SMHiggsToZZTo4L.root": "Events"},
    schemaclass=SlimNanoAODSchema,
    metadata={"dataset": "SMHiggsToZZTo4L"},
    mode="virtual",
    access_log=access_log,
).events()
events.fields

**Quizlet 6.1 — electrons too.** Compute the invariant mass of the two leading electrons, in events with at least two electrons.
Then look at `access_log`: which columns were read? What extra columns appear to be read past what you minimally need?

**Quizlet 6.2 - try virtual mode**

Create a `virtual_events` array using that separate mode, with an `access_log_virtual`, and compare to eager

In [ ]:
access_log_virtual = []
virtual_events = None # fill in from here

The selected leptons and the selection from the slides, to start Quizlet 6.2 from:

In [ ]:
muons = events.Muon[
    (events.Muon.pt > 5) & (abs(events.Muon.eta) < 2.4) & (events.Muon.pfRelIso04_all < 0.4)
]
electrons = events.Electron[
    (events.Electron.pt > 7) & (abs(events.Electron.eta) < 2.5) & (events.Electron.pfRelIso03_all < 0.4)
]

selection = PackedSelection()
selection.add("fourLeptons", ak.num(muons) + ak.num(electrons) == 4)
selection.add("chargeNeutral", ak.sum(muons.charge, axis=1) + ak.sum(electrons.charge, axis=1) == 0)
selection.add("sameFlavourPairs", ak.num(muons) % 2 == 0)
selection.add("leadingPt20", ak.any(muons.pt > 20, axis=1) | ak.any(electrons.pt > 20, axis=1))

cuts = list(selection.names)
selection.cutflow(*cuts).print()

**Quizlet 6.3 — one more cut.** Add a cut `"fourMuons"` (exactly four selected muons) to `selection`,
and print the cutflow with it as the last step. How many $4\mu$ candidates are left?